In [ ]:
import os
import glob
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddingsfrom langchain_openai import OpenAIEmbeddings


/Users/parthapratimdas/RAG-basics/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/parthapratimdas/RAG-basics/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(override=True)
MODEL = "deepseek/deepseek-r1:free"


def _project_root() -> Path:
    """Repo root containing knowledge-base/ (works in notebooks and scripts)."""
    try:
        p = Path(__file__).resolve().parent.parent
        if (p / "knowledge-base").is_dir():
            return p
    except NameError:
        pass
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "knowledge-base").is_dir():
            return p
    raise FileNotFoundError(
        "Could not find knowledge-base/. Open the notebook from the repo or run from project root."
    )


base_path = _project_root()
DB_NAME = str(base_path / "vector_db")
KNOWLEDGE_BASE = str(base_path / "knowledge-base")
print("Repo root:", base_path)
print("KNOWLEDGE_BASE:", KNOWLEDGE_BASE, "| exists:", Path(KNOWLEDGE_BASE).is_dir())

In [ ]:
load_dotenv(override=True)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


In [ ]:
from pathlib import Path
import os
kb = Path("...")  # whatever you use as KNOWLEDGE_BASE
print("KNOWLEDGE_BASE:", kb.resolve())
print("exists:", kb.exists())
print("subdirs:", list(kb.glob("*")))

KNOWLEDGE_BASE: /Users/parthapratimdas/RAG-basics/implementation/...
exists: False
subdirs: []


In [15]:
import os
import glob
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, TextLoader

def fetch_documents():
    # Use Path to ensure cross-platform compatibility (Windows vs Mac)
    path_pattern = str(Path(KNOWLEDGE_BASE) / "*")
    folders = glob.glob(path_pattern)
    
    documents = []
    
    for folder in folders:
        # CRITICAL: Only proceed if it's actually a folder
        if os.path.isdir(folder):
            doc_type = os.path.basename(folder)
            print(f"Scanning folder: {doc_type}...") # Debugging line
            
            loader = DirectoryLoader(
                folder, 
                glob="**/*.md", 
                loader_cls=TextLoader, 
                loader_kwargs={"encoding": "utf-8"},
                recursive=True # Ensure it digs deep
            )
            
            folder_docs = loader.load()
            
            for doc in folder_docs:
                doc.metadata["doc_type"] = doc_type
                # Also ensure the 'source' metadata is a clean string
                doc.metadata["source"] = os.path.basename(doc.metadata.get("source", ""))
                documents.append(doc)
                
    print(f"Total documents loaded: {len(documents)}")
    return documents

In [16]:
def create_chunks(documents):
    # text_splitter = MarkdownTextSplitter()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=200)
    chunks = text_splitter.split_documents(documents)
    return chunks

In [ ]:
def create_embeddings(chunks):
    if os.path.exists(DB_NAME):
        Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()

    vectorstore = Chroma.from_documents(
        documents=chunks, embedding=embeddings, persist_directory=DB_NAME
    )

    collection = vectorstore._collection
    count = collection.count()

    sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
    dimensions = len(sample_embedding)
    print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")
    return vectorstore

In [ ]:
if __name__ == "__main__":
    documents = fetch_documents()
    chunks = create_chunks(documents)
    create_embeddings(chunks)
    print("Ingestion complete")


Total documents loaded: 0


ValueError: Expected Embeddings to be non-empty list or numpy array, got [] in upsert.